# Hypothesis Testing Framework

Tests the 9 hypotheses from `strategy_summary.ipynb` using the existing
`analysis/` modules and new `strategy/` backtesting infrastructure.

**Prerequisites:** Data must be fetched (`scripts/fetch_data.py`) and
preprocessed (`preprocess_all_tickers()`) before running.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import itertools
import warnings

import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from tqdm.auto import tqdm

from utils.config import (
    CONFIG,
    get_all_sectors,
    get_all_tickers,
    get_tickers_by_sector,
)
from analysis.preprocessing import load_processed
from analysis.cointegration import (
    find_cointegrated_pairs,
    find_cointegrated_pairs_rolling,
    test_cointegration,
    track_pair_stability,
)
from strategy.backtester import PairBacktester, PortfolioBacktester
from strategy.metrics import (
    Trade,
    calculate_metrics,
    build_equity_curve,
    bootstrap_sharpe,
)

warnings.filterwarnings("ignore", category=FutureWarning)
print(f"Project root: {project_root}")
print(f"Universe:     {len(get_all_tickers())} tickers, {len(get_all_sectors())} sectors")

## Shared Setup

Derive date ranges and periods from the available data rather than hardcoding.

In [ ]:
# --- Discover available data range from a representative ticker ---
all_tickers = get_all_tickers()
sample_df = load_processed(all_tickers[0], "daily")
DATA_START = sample_df["timestamp"].min()
DATA_END = sample_df["timestamp"].max()
data_days = (DATA_END - DATA_START).days
print(f"Data range: {DATA_START.date()} to {DATA_END.date()} ({data_days} calendar days)")

# --- Define formation / trading periods as fractions of available data ---
# First 25% = formation, next 25% = trading (for basic backtests)
from datetime import timedelta
quarter = timedelta(days=data_days // 4)

FORMATION_START = DATA_START.strftime("%Y-%m-%d")
FORMATION_END = (DATA_START + quarter).strftime("%Y-%m-%d")
TRADING_START = FORMATION_END
TRADING_END = (DATA_START + 2 * quarter).strftime("%Y-%m-%d")

# For IS/OOS split (H4): 60% IS, 40% OOS of total range
is_days = int(data_days * 0.6)
IS_START = DATA_START.strftime("%Y-%m-%d")
IS_END = (DATA_START + timedelta(days=is_days)).strftime("%Y-%m-%d")
OOS_START = IS_END
OOS_END = DATA_END.strftime("%Y-%m-%d")

# Default timeframe for hypothesis tests
DEFAULT_TF = "daily"

print(f"Formation: {FORMATION_START} to {FORMATION_END}")
print(f"Trading:   {TRADING_START} to {TRADING_END}")
print(f"IS period: {IS_START} to {IS_END}")
print(f"OOS period:{OOS_START} to {OOS_END}")

In [ ]:
# --- Discover cointegrated pairs on the formation period ---
formation_pairs = find_cointegrated_pairs(
    timeframe=DEFAULT_TF,
    start_date=FORMATION_START,
    end_date=FORMATION_END,
)
print(f"Found {len(formation_pairs)} cointegrated pairs on {DEFAULT_TF} data")
print(f"Sectors represented: {formation_pairs['sector'].n_unique()}")
formation_pairs.head(10)

---
## Hypothesis 1: Within-Sector Cointegration

**H\u2080:** Within-sector pairs are no more likely to be cointegrated than cross-sector pairs.  
**H\u2081:** Within-sector pairs exhibit significantly higher cointegration rates.

**Method:** Compare ADF pass rates (p < 0.05) for within-sector vs. random cross-sector pairs using a chi-square test.

In [ ]:
# --- Within-sector pass rate (already computed) ---
within_pairs_tested = 0
within_pairs_passed = len(formation_pairs)

# Count total within-sector combinations to get the denominator
for sector in get_all_sectors():
    tickers = get_tickers_by_sector(sector)
    within_pairs_tested += len(list(itertools.combinations(tickers, 2)))

within_pass_rate = within_pairs_passed / within_pairs_tested if within_pairs_tested > 0 else 0
print(f"Within-sector: {within_pairs_passed}/{within_pairs_tested} passed ({within_pass_rate:.1%})")

In [ ]:
# --- Cross-sector: sample random pairs from different sectors ---
rng = np.random.default_rng(42)
n_cross_samples = max(within_pairs_tested, 500)  # at least as many as within-sector

cross_passed = 0
cross_tested = 0
cross_p_values = []

# Build sector-to-tickers lookup
sectors = get_all_sectors()
sector_tickers = {s: get_tickers_by_sector(s) for s in sectors}

# Pre-load prices for efficiency
price_cache = {}
for t in tqdm(all_tickers, desc="Loading prices"):
    try:
        df = load_processed(t, DEFAULT_TF, start_date=FORMATION_START, end_date=FORMATION_END)
        if len(df) >= 60:
            price_cache[t] = df.select(["timestamp", "close"]).rename({"close": t})
    except FileNotFoundError:
        pass

attempts = 0
max_attempts = n_cross_samples * 3  # guard against infinite loop

while cross_tested < n_cross_samples and attempts < max_attempts:
    attempts += 1
    # Pick two tickers from different sectors
    s1, s2 = rng.choice(sectors, size=2, replace=False)
    t1 = rng.choice(sector_tickers[s1])
    t2 = rng.choice(sector_tickers[s2])
    if t1 not in price_cache or t2 not in price_cache:
        continue

    merged = price_cache[t1].join(price_cache[t2], on="timestamp", how="inner")
    if len(merged) < 60:
        continue

    pa = merged[t1].to_numpy().astype(np.float64)
    pb = merged[t2].to_numpy().astype(np.float64)
    try:
        result = test_cointegration(pa, pb)
    except Exception:
        continue

    cross_tested += 1
    cross_p_values.append(result["p_value"])
    if result["p_value"] <= CONFIG["coint_p_value_threshold"]:
        cross_passed += 1

cross_pass_rate = cross_passed / cross_tested if cross_tested > 0 else 0
print(f"Cross-sector:  {cross_passed}/{cross_tested} passed ({cross_pass_rate:.1%})")

In [ ]:
# --- Chi-square test and odds ratio ---
contingency = np.array([
    [within_pairs_passed, within_pairs_tested - within_pairs_passed],
    [cross_passed, cross_tested - cross_passed],
])
chi2, chi2_p, dof, expected = stats.chi2_contingency(contingency)

# Odds ratio
a, b = contingency[0]
c, d = contingency[1]
odds_ratio = (a * d) / (b * c) if (b * c) > 0 else float("inf")
# 95% CI for log odds ratio
log_or = np.log(odds_ratio) if odds_ratio > 0 and odds_ratio != float("inf") else 0
se_log_or = np.sqrt(1/max(a,1) + 1/max(b,1) + 1/max(c,1) + 1/max(d,1))
ci_lower = np.exp(log_or - 1.96 * se_log_or)
ci_upper = np.exp(log_or + 1.96 * se_log_or)

print("=== H1: Within-Sector Cointegration ===")
print(f"Within-sector pass rate: {within_pass_rate:.1%}")
print(f"Cross-sector pass rate:  {cross_pass_rate:.1%}")
print(f"Chi-square statistic:    {chi2:.2f} (p = {chi2_p:.4e})")
print(f"Odds ratio:              {odds_ratio:.2f} (95% CI: [{ci_lower:.2f}, {ci_upper:.2f}])")
print()
if chi2_p < 0.05:
    print("RESULT: Reject H0 — within-sector pairs are significantly more cointegrated.")
else:
    print("RESULT: Fail to reject H0 — no significant difference found.")

In [ ]:
# --- Visualize pass rates by sector ---
sector_stats = []
for sector in get_all_sectors():
    n_sector = len(formation_pairs.filter(pl.col("sector") == sector))
    tickers = get_tickers_by_sector(sector)
    n_possible = len(list(itertools.combinations(tickers, 2)))
    sector_stats.append({
        "sector": sector,
        "pairs_passed": n_sector,
        "pairs_tested": n_possible,
        "pass_rate": n_sector / n_possible if n_possible > 0 else 0,
    })

sector_df = pl.DataFrame(sector_stats).sort("pass_rate", descending=True)
fig = px.bar(
    sector_df.to_pandas(), x="sector", y="pass_rate",
    text="pairs_passed",
    title="H1: Cointegration Pass Rate by Sector",
    labels={"pass_rate": "Pass Rate", "sector": "Sector"},
)
fig.add_hline(y=cross_pass_rate, line_dash="dash", line_color="red",
              annotation_text=f"Cross-sector baseline: {cross_pass_rate:.1%}")
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

---
## Hypothesis 2: Z-Score Entry Threshold Optimization

**H\u2080:** Choice of z-score entry threshold does not significantly affect risk-adjusted returns.  
**H\u2081:** There exists an optimal threshold that maximizes Sharpe ratio.

**Method:** Grid search z\_entry over {1.5, 2.0, 2.5, 3.0, 3.5}, backtest each on the trading period.

In [ ]:
z_entry_grid = [1.5, 2.0, 2.5, 3.0, 3.5]
h2_results = []

for z_entry in tqdm(z_entry_grid, desc="H2: z_entry grid"):
    bt = PortfolioBacktester(
        z_entry=z_entry,
    )
    result = bt.backtest(
        formation_pairs,
        timeframe=DEFAULT_TF,
        start_date=TRADING_START,
        end_date=TRADING_END,
    )
    m = result.metrics
    bs = bootstrap_sharpe(result.trades, seed=42)
    h2_results.append({
        "z_entry": z_entry,
        "sharpe": m["sharpe_ratio"],
        "sharpe_ci_lo": bs["ci_lower"],
        "sharpe_ci_hi": bs["ci_upper"],
        "num_trades": m["num_trades"],
        "win_rate": m["win_rate"],
        "avg_pnl": m["avg_pnl_net"],
        "max_dd_pct": m["max_drawdown_pct"],
        "total_return_pct": m["total_return_pct"],
    })

h2_df = pl.DataFrame(h2_results)
print(h2_df)

In [ ]:
# --- Visualize: Sharpe ratio vs z_entry with bootstrap CI ---
h2_pd = h2_df.to_pandas()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=h2_pd["z_entry"], y=h2_pd["sharpe"],
    mode="lines+markers", name="Sharpe Ratio",
    error_y=dict(
        type="data",
        symmetric=False,
        array=(h2_pd["sharpe_ci_hi"] - h2_pd["sharpe"]).tolist(),
        arrayminus=(h2_pd["sharpe"] - h2_pd["sharpe_ci_lo"]).tolist(),
    ),
))
fig.update_layout(
    title="H2: Sharpe Ratio vs Z-Score Entry Threshold",
    xaxis_title="Z-Score Entry Threshold",
    yaxis_title="Sharpe Ratio",
)
fig.show()

# Trade frequency and win rate
fig2 = make_subplots(specs=[[{"secondary_y": True}]])
fig2.add_trace(go.Bar(x=h2_pd["z_entry"], y=h2_pd["num_trades"], name="Num Trades"), secondary_y=False)
fig2.add_trace(go.Scatter(x=h2_pd["z_entry"], y=h2_pd["win_rate"], name="Win Rate", mode="lines+markers"), secondary_y=True)
fig2.update_layout(title="H2: Trade Frequency and Win Rate vs Threshold")
fig2.update_yaxes(title_text="Number of Trades", secondary_y=False)
fig2.update_yaxes(title_text="Win Rate", tickformat=".0%", secondary_y=True)
fig2.show()

best_z = h2_pd.loc[h2_pd["sharpe"].idxmax()]
print(f"\nOptimal z_entry = {best_z['z_entry']} (Sharpe = {best_z['sharpe']:.2f})")

---
## Hypothesis 3: Transaction Cost Sensitivity

**H\u2080:** Strategy remains profitable across 10–30 bps per leg.  
**H\u2081:** Strategy is highly sensitive to costs; profitability disappears above ~25 bps.

**Method:** Grid search transaction\_cost\_bps over {5, 10, 15, 20, 25, 30}.

In [ ]:
cost_grid = [5, 10, 15, 20, 25, 30]
h3_results = []

for cost_bps in tqdm(cost_grid, desc="H3: cost grid"):
    bt = PortfolioBacktester(
        transaction_cost_bps=cost_bps,
    )
    result = bt.backtest(
        formation_pairs,
        timeframe=DEFAULT_TF,
        start_date=TRADING_START,
        end_date=TRADING_END,
    )
    m = result.metrics
    h3_results.append({
        "cost_bps": cost_bps,
        "sharpe": m["sharpe_ratio"],
        "total_return_pct": m["total_return_pct"],
        "total_pnl_gross": m["total_pnl_gross"],
        "total_pnl_net": m["total_pnl_net"],
        "win_rate": m["win_rate"],
        "profit_factor": m["profit_factor"],
        "num_trades": m["num_trades"],
    })

h3_df = pl.DataFrame(h3_results)
print(h3_df)

In [ ]:
h3_pd = h3_df.to_pandas()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=h3_pd["cost_bps"], y=h3_pd["sharpe"],
    mode="lines+markers", name="Sharpe Ratio",
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=h3_pd["cost_bps"], y=h3_pd["total_return_pct"],
    mode="lines+markers", name="Total Return %",
), secondary_y=True)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(title="H3: Performance vs Transaction Cost")
fig.update_xaxes(title_text="Transaction Cost (bps per leg)")
fig.update_yaxes(title_text="Sharpe Ratio", secondary_y=False)
fig.update_yaxes(title_text="Total Return %", secondary_y=True)
fig.show()

# Estimate breakeven cost via linear interpolation
if h3_pd["sharpe"].iloc[0] > 0 and h3_pd["sharpe"].iloc[-1] < 0:
    from scipy.interpolate import interp1d
    f = interp1d(h3_pd["sharpe"], h3_pd["cost_bps"])
    breakeven = float(f(0))
    print(f"Estimated breakeven cost: {breakeven:.1f} bps per leg")
elif h3_pd["sharpe"].min() > 0:
    print(f"Strategy is profitable at all tested cost levels (max {cost_grid[-1]} bps)")
else:
    print(f"Strategy is unprofitable at all tested cost levels")

# Sensitivity coefficient: delta Sharpe per bps
slope, intercept, r, p, se = stats.linregress(h3_pd["cost_bps"], h3_pd["sharpe"])
print(f"Sensitivity: {slope:.4f} Sharpe / bps (R² = {r**2:.3f})")

---
## Hypothesis 4: In-Sample vs Out-of-Sample Degradation

**H\u2080:** IS-optimized parameters perform equally well OOS.  
**H\u2081:** Significant OOS degradation due to overfitting.

**Method:** Optimize z\_entry on IS data (60%), evaluate on OOS (40%), measure degradation.

In [ ]:
# --- In-sample pair discovery ---
# Split IS into formation half and trading half
from datetime import datetime
is_start_dt = datetime.fromisoformat(IS_START)
is_end_dt = datetime.fromisoformat(IS_END)
is_mid = is_start_dt + (is_end_dt - is_start_dt) / 2
IS_FORM_END = is_mid.strftime("%Y-%m-%d")
IS_TRADE_START = IS_FORM_END

print(f"IS formation: {IS_START} to {IS_FORM_END}")
print(f"IS trading:   {IS_TRADE_START} to {IS_END}")
print(f"OOS period:   {OOS_START} to {OOS_END}")

is_pairs = find_cointegrated_pairs(
    timeframe=DEFAULT_TF,
    start_date=IS_START,
    end_date=IS_FORM_END,
)
print(f"IS pairs found: {len(is_pairs)}")

In [ ]:
# --- Grid search on IS trading period ---
h4_z_grid = [1.5, 2.0, 2.5, 3.0, 3.5]
is_sharpes = {}

for z_entry in tqdm(h4_z_grid, desc="H4: IS optimization"):
    bt = PortfolioBacktester(z_entry=z_entry)
    result = bt.backtest(
        is_pairs,
        timeframe=DEFAULT_TF,
        start_date=IS_TRADE_START,
        end_date=IS_END,
    )
    is_sharpes[z_entry] = result.metrics["sharpe_ratio"]
    print(f"  z_entry={z_entry}: Sharpe={result.metrics['sharpe_ratio']:.3f}, trades={result.metrics['num_trades']}")

best_z_is = max(is_sharpes, key=is_sharpes.get)
print(f"\nBest IS z_entry = {best_z_is} (Sharpe = {is_sharpes[best_z_is]:.3f})")

In [ ]:
# --- OOS evaluation with IS-optimal parameters ---
# Re-discover pairs on OOS formation window (first half of OOS)
oos_start_dt = datetime.fromisoformat(OOS_START)
oos_end_dt = datetime.fromisoformat(OOS_END)
oos_mid = oos_start_dt + (oos_end_dt - oos_start_dt) / 2
OOS_FORM_END = oos_mid.strftime("%Y-%m-%d")
OOS_TRADE_START = OOS_FORM_END

oos_pairs = find_cointegrated_pairs(
    timeframe=DEFAULT_TF,
    start_date=OOS_START,
    end_date=OOS_FORM_END,
)
print(f"OOS pairs found: {len(oos_pairs)}")

# Run IS-best AND default parameters on OOS
h4_comparison = []
for label, z_val, pairs, trade_start, trade_end in [
    ("IS (best z)", best_z_is, is_pairs, IS_TRADE_START, IS_END),
    ("OOS (IS-best z)", best_z_is, oos_pairs, OOS_TRADE_START, OOS_END),
    ("OOS (default z)", CONFIG["z_score_entry"], oos_pairs, OOS_TRADE_START, OOS_END),
]:
    bt = PortfolioBacktester(z_entry=z_val)
    result = bt.backtest(pairs, timeframe=DEFAULT_TF, start_date=trade_start, end_date=trade_end)
    m = result.metrics
    bs = bootstrap_sharpe(result.trades, seed=42)
    h4_comparison.append({
        "period": label,
        "z_entry": z_val,
        "sharpe": m["sharpe_ratio"],
        "sharpe_ci_lo": bs["ci_lower"],
        "sharpe_ci_hi": bs["ci_upper"],
        "return_pct": m["total_return_pct"],
        "max_dd_pct": m["max_drawdown_pct"],
        "win_rate": m["win_rate"],
        "num_trades": m["num_trades"],
    })

h4_df = pl.DataFrame(h4_comparison)
print(h4_df)

# Degradation
sharpe_is = h4_df.filter(pl.col("period") == "IS (best z)")["sharpe"][0]
sharpe_oos = h4_df.filter(pl.col("period") == "OOS (IS-best z)")["sharpe"][0]
if sharpe_is != 0:
    degradation = (sharpe_is - sharpe_oos) / abs(sharpe_is) * 100
    print(f"\nSharpe degradation IS → OOS: {degradation:.1f}%")
    if abs(degradation) < 30:
        print("RESULT: Moderate degradation (<30%) — parameters reasonably robust.")
    elif degradation > 50:
        print("RESULT: Severe degradation (>50%) — likely overfitting.")
    else:
        print(f"RESULT: {degradation:.0f}% degradation — some overfitting present.")
else:
    print("Cannot compute degradation: IS Sharpe is zero.")

---
## Hypothesis 5: Half-Life Stability Over Time

**H\u2080:** Half-life estimates are unstable over time (high CV).  
**H\u2081:** Half-life is relatively stable for strong pairs, making it a useful filter.

**Method:** Use `track_pair_stability()` with non-overlapping windows. Compare CV distributions and backtest stable vs unstable pairs.

In [ ]:
# --- Build non-overlapping quarterly windows from available data ---
from dateutil.relativedelta import relativedelta

window_months = 3  # quarterly
current = datetime.fromisoformat(FORMATION_START)
end_dt = datetime.fromisoformat(OOS_END)
windows = []
while current + relativedelta(months=window_months) <= end_dt:
    w_end = current + relativedelta(months=window_months)
    windows.append((current.strftime("%Y-%m-%d"), w_end.strftime("%Y-%m-%d")))
    current = w_end

print(f"Built {len(windows)} non-overlapping windows:")
for s, e in windows:
    print(f"  {s} to {e}")

In [ ]:
# --- Track stability for top formation pairs ---
n_pairs_to_track = min(20, len(formation_pairs))  # limit for speed
stability_results = []

for row in tqdm(
    formation_pairs.head(n_pairs_to_track).iter_rows(named=True),
    total=n_pairs_to_track,
    desc="H5: tracking stability",
):
    stab = track_pair_stability(
        row["ticker_a"], row["ticker_b"],
        windows=windows,
        timeframe=DEFAULT_TF,
    )
    # Calculate CV of half-life across windows where pair is cointegrated
    cointegrated = stab.filter(pl.col("cointegrated"))
    if len(cointegrated) >= 2:
        hl = cointegrated["half_life"].drop_nulls()
        if len(hl) >= 2 and hl.mean() > 0:
            cv = float(hl.std() / hl.mean())
        else:
            cv = float("inf")
    else:
        cv = float("inf")

    stability_results.append({
        "ticker_a": row["ticker_a"],
        "ticker_b": row["ticker_b"],
        "p_value": row["p_value"],
        "half_life_cv": cv,
        "n_cointegrated_windows": len(cointegrated) if len(cointegrated) >= 0 else 0,
        "n_windows": len(windows),
        "stable": cv < 0.5,
    })

h5_df = pl.DataFrame(stability_results)
print(h5_df.sort("half_life_cv"))

In [ ]:
# --- Compare backtest performance: stable vs unstable pairs ---
stable_pairs_list = h5_df.filter(pl.col("stable"))
unstable_pairs_list = h5_df.filter(~pl.col("stable"))

print(f"Stable pairs (CV < 0.5): {len(stable_pairs_list)}")
print(f"Unstable pairs (CV >= 0.5): {len(unstable_pairs_list)}")

h5_comparison = []
for label, pair_set in [("Stable", stable_pairs_list), ("Unstable", unstable_pairs_list)]:
    if len(pair_set) == 0:
        h5_comparison.append({"group": label, "sharpe": 0, "win_rate": 0, "num_trades": 0, "return_pct": 0})
        continue
    # Filter formation_pairs to only include these pairs
    pair_keys = set(
        (r["ticker_a"], r["ticker_b"]) for r in pair_set.iter_rows(named=True)
    )
    filtered = formation_pairs.filter(
        pl.struct(["ticker_a", "ticker_b"]).map_elements(
            lambda r: (r["ticker_a"], r["ticker_b"]) in pair_keys,
            return_dtype=pl.Boolean,
        )
    )
    if len(filtered) == 0:
        h5_comparison.append({"group": label, "sharpe": 0, "win_rate": 0, "num_trades": 0, "return_pct": 0})
        continue

    bt = PortfolioBacktester(max_pairs=len(filtered))
    result = bt.backtest(
        filtered,
        timeframe=DEFAULT_TF,
        start_date=TRADING_START,
        end_date=TRADING_END,
        max_pairs_to_trade=len(filtered),
    )
    m = result.metrics
    h5_comparison.append({
        "group": label,
        "sharpe": m["sharpe_ratio"],
        "win_rate": m["win_rate"],
        "num_trades": m["num_trades"],
        "return_pct": m["total_return_pct"],
    })

h5_comp_df = pl.DataFrame(h5_comparison)
print("\n=== H5: Stable vs Unstable Pairs ===")
print(h5_comp_df)

# Visualize CV distribution
finite_cv = h5_df.filter(pl.col("half_life_cv") < 10)  # exclude inf for plotting
if len(finite_cv) > 0:
    fig = px.histogram(
        finite_cv.to_pandas(), x="half_life_cv", nbins=15,
        title="H5: Distribution of Half-Life CV Across Pairs",
        labels={"half_life_cv": "Coefficient of Variation"},
    )
    fig.add_vline(x=0.5, line_dash="dash", line_color="red",
                  annotation_text="Stability threshold (0.5)")
    fig.show()

---
## Hypothesis 6: Stop Loss Effectiveness

**H\u2080:** Stop loss at z = 4.0 does not improve risk-adjusted returns vs no stop.  
**H\u2081:** Stop loss significantly reduces tail risk and improves Sharpe.

**Method:** Compare backtests with z\_stop in {3.0, 3.5, 4.0, 4.5, 5.0, inf}.

In [ ]:
z_stop_grid = [3.0, 3.5, 4.0, 4.5, 5.0, float("inf")]
h6_results = []

for z_stop in tqdm(z_stop_grid, desc="H6: z_stop grid"):
    bt = PortfolioBacktester(z_stop=z_stop)
    result = bt.backtest(
        formation_pairs,
        timeframe=DEFAULT_TF,
        start_date=TRADING_START,
        end_date=TRADING_END,
    )
    m = result.metrics
    h6_results.append({
        "z_stop": z_stop if z_stop != float("inf") else 999,
        "z_stop_label": str(z_stop) if z_stop != float("inf") else "No Stop",
        "sharpe": m["sharpe_ratio"],
        "sortino": m["sortino_ratio"],
        "max_dd_pct": m["max_drawdown_pct"],
        "var_95": m["var_95"],
        "cvar_95": m["cvar_95"],
        "tail_ratio": m["tail_ratio"],
        "pct_stop_loss": m["pct_stop_loss"],
        "pct_mean_reversion": m["pct_mean_reversion"],
        "win_rate": m["win_rate"],
        "num_trades": m["num_trades"],
    })

h6_df = pl.DataFrame(h6_results)
print(h6_df)

In [ ]:
h6_pd = h6_df.to_pandas()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Risk-Adjusted Returns", "Tail Risk Metrics"])

fig.add_trace(go.Bar(x=h6_pd["z_stop_label"], y=h6_pd["sharpe"], name="Sharpe"), row=1, col=1)
fig.add_trace(go.Bar(x=h6_pd["z_stop_label"], y=h6_pd["sortino"], name="Sortino"), row=1, col=1)

fig.add_trace(go.Bar(x=h6_pd["z_stop_label"], y=h6_pd["max_dd_pct"], name="Max DD %"), row=1, col=2)
fig.add_trace(go.Bar(x=h6_pd["z_stop_label"], y=h6_pd["cvar_95"], name="CVaR 95"), row=1, col=2)

fig.update_layout(title="H6: Stop Loss Effectiveness", height=400)
fig.show()

# Exit reason breakdown
fig2 = go.Figure()
fig2.add_trace(go.Bar(x=h6_pd["z_stop_label"], y=h6_pd["pct_mean_reversion"], name="Mean Reversion"))
fig2.add_trace(go.Bar(x=h6_pd["z_stop_label"], y=h6_pd["pct_stop_loss"], name="Stop Loss"))
fig2.update_layout(
    title="H6: Exit Reason Breakdown by Stop Level",
    barmode="stack", yaxis_tickformat=".0%",
)
fig2.show()

---
## Hypothesis 7: Portfolio Diversification Benefits

**H\u2080:** Trading 10 pairs does not reduce portfolio volatility vs 3–5 pairs.  
**H\u2081:** More pairs = lower volatility and higher Sharpe (diversification).

**Method:** Compare max\_pairs in {3, 5, 10, 15, 20}.

In [ ]:
# Build grid up to the number of available pairs
max_available = len(formation_pairs)
pairs_grid = sorted(set(p for p in [3, 5, 10, 15, 20] if p <= max_available))
if max_available not in pairs_grid and max_available > 0:
    pairs_grid.append(max_available)
    pairs_grid.sort()

h7_results = []
for n_pairs in tqdm(pairs_grid, desc="H7: diversification"):
    bt = PortfolioBacktester(max_pairs=n_pairs)
    result = bt.backtest(
        formation_pairs,
        timeframe=DEFAULT_TF,
        start_date=TRADING_START,
        end_date=TRADING_END,
        max_pairs_to_trade=n_pairs,
    )
    m = result.metrics

    # Calculate per-pair P&L correlation (if enough trades)
    pair_pnls = {}
    for t in result.trades:
        key = (t.ticker_a, t.ticker_b)
        pair_pnls.setdefault(key, []).append(t.pnl_net)

    avg_corr = np.nan
    if len(pair_pnls) >= 2:
        # Pad to same length for correlation
        max_len = max(len(v) for v in pair_pnls.values())
        padded = {k: v + [0.0] * (max_len - len(v)) for k, v in pair_pnls.items()}
        corr_matrix = np.corrcoef(list(padded.values()))
        # Average off-diagonal correlations
        n = corr_matrix.shape[0]
        if n >= 2:
            off_diag = corr_matrix[np.triu_indices(n, k=1)]
            avg_corr = float(np.nanmean(off_diag))

    h7_results.append({
        "n_pairs": n_pairs,
        "sharpe": m["sharpe_ratio"],
        "max_dd_pct": m["max_drawdown_pct"],
        "total_return_pct": m["total_return_pct"],
        "num_trades": m["num_trades"],
        "avg_pair_corr": avg_corr,
    })

h7_df = pl.DataFrame(h7_results)
print(h7_df)

In [ ]:
h7_pd = h7_df.to_pandas()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Sharpe vs Pairs", "Max Drawdown vs Pairs"])
fig.add_trace(go.Scatter(
    x=h7_pd["n_pairs"], y=h7_pd["sharpe"],
    mode="lines+markers", name="Sharpe",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=h7_pd["n_pairs"], y=h7_pd["max_dd_pct"],
    mode="lines+markers", name="Max DD %",
), row=1, col=2)
fig.update_layout(title="H7: Portfolio Diversification Benefits", height=400)
fig.show()

# Marginal benefit
if len(h7_pd) >= 2:
    h7_pd["marginal_sharpe"] = h7_pd["sharpe"].diff()
    print("\nMarginal Sharpe improvement:")
    for _, row in h7_pd.iterrows():
        ms = f"{row['marginal_sharpe']:+.3f}" if not np.isnan(row.get('marginal_sharpe', np.nan)) else 'N/A'
        print(f"  {int(row['n_pairs']):3d} pairs: Sharpe={row['sharpe']:.3f}  Δ={ms}")

---
## Hypothesis 8: Walk-Forward Robustness

**H\u2080:** Strategy performance is unstable across walk-forward windows.  
**H\u2081:** Consistent risk-adjusted returns, indicating genuine alpha.

**Method:** Rolling 6-month formation / 3-month OOS using `find_cointegrated_pairs_rolling()`.

In [ ]:
# --- Walk-forward windows ---
wf_formation_months = 6
wf_trading_months = 3
wf_step_months = wf_trading_months  # non-overlapping OOS periods

# Build formation windows
wf_start = datetime.fromisoformat(FORMATION_START)
wf_final = datetime.fromisoformat(OOS_END)

wf_windows = []  # (form_start, form_end, trade_start, trade_end)
current = wf_start
while True:
    form_end = current + relativedelta(months=wf_formation_months)
    trade_start = form_end
    trade_end = trade_start + relativedelta(months=wf_trading_months)
    if trade_end > wf_final:
        break
    wf_windows.append((
        current.strftime("%Y-%m-%d"),
        form_end.strftime("%Y-%m-%d"),
        trade_start.strftime("%Y-%m-%d"),
        trade_end.strftime("%Y-%m-%d"),
    ))
    current += relativedelta(months=wf_step_months)

print(f"Walk-forward: {len(wf_windows)} periods")
for i, (fs, fe, ts, te) in enumerate(wf_windows):
    print(f"  Period {i+1}: Formation {fs}→{fe}, Trading {ts}→{te}")

In [ ]:
h8_results = []

for i, (form_start, form_end, trade_start, trade_end) in enumerate(
    tqdm(wf_windows, desc="H8: walk-forward")
):
    # Discover pairs on formation window
    wf_pairs = find_cointegrated_pairs(
        timeframe=DEFAULT_TF,
        start_date=form_start,
        end_date=form_end,
    )
    if len(wf_pairs) == 0:
        h8_results.append({
            "period": i + 1,
            "form_start": form_start,
            "trade_start": trade_start,
            "trade_end": trade_end,
            "n_pairs": 0,
            "sharpe": 0.0,
            "return_pct": 0.0,
            "max_dd_pct": 0.0,
            "num_trades": 0,
            "win_rate": 0.0,
        })
        continue

    bt = PortfolioBacktester()
    result = bt.backtest(
        wf_pairs,
        timeframe=DEFAULT_TF,
        start_date=trade_start,
        end_date=trade_end,
    )
    m = result.metrics
    h8_results.append({
        "period": i + 1,
        "form_start": form_start,
        "trade_start": trade_start,
        "trade_end": trade_end,
        "n_pairs": len(wf_pairs),
        "sharpe": m["sharpe_ratio"],
        "return_pct": m["total_return_pct"],
        "max_dd_pct": m["max_drawdown_pct"],
        "num_trades": m["num_trades"],
        "win_rate": m["win_rate"],
    })

h8_df = pl.DataFrame(h8_results)
print(h8_df)

In [ ]:
h8_pd = h8_df.to_pandas()

# Summary statistics
sharpes = h8_pd["sharpe"]
print("=== H8: Walk-Forward Summary ===")
print(f"Mean OOS Sharpe:         {sharpes.mean():.3f}")
print(f"Std OOS Sharpe:          {sharpes.std():.3f}")
print(f"% positive Sharpe:       {(sharpes > 0).mean():.0%}")
print(f"% Sharpe > 1.0:          {(sharpes > 1.0).mean():.0%}")
print(f"Worst period Sharpe:     {sharpes.min():.3f}")
print(f"Best period Sharpe:      {sharpes.max():.3f}")
if sharpes.std() > 0:
    print(f"Information ratio:       {sharpes.mean() / sharpes.std():.3f}")

# Time series plot
fig = go.Figure()
fig.add_trace(go.Bar(
    x=h8_pd["trade_start"], y=h8_pd["sharpe"],
    marker_color=["green" if s > 0 else "red" for s in h8_pd["sharpe"]],
    name="OOS Sharpe",
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.add_hline(y=sharpes.mean(), line_dash="dot", line_color="blue",
              annotation_text=f"Mean: {sharpes.mean():.2f}")
fig.update_layout(
    title="H8: Walk-Forward OOS Sharpe Ratios",
    xaxis_title="Trading Period Start",
    yaxis_title="Sharpe Ratio",
)
fig.show()

---
## Hypothesis 9: P-Value as Pair Quality Signal

**H\u2080:** ADF p-value does not predict pair profitability.  
**H\u2081:** Lower p-values → higher risk-adjusted returns.

**Method:** Sort pairs into quintiles by p-value, backtest each quintile separately.

In [ ]:
# --- Assign p-value quintiles ---
n_available = len(formation_pairs)
n_quintiles = min(5, n_available)  # handle case with very few pairs

if n_available < n_quintiles:
    print(f"Only {n_available} pairs — cannot form {n_quintiles} quintiles. Reducing.")
    n_quintiles = max(2, n_available)

# Add quintile labels (Q1 = lowest p-value = strongest cointegration)
pairs_with_q = formation_pairs.with_row_index("rank").with_columns(
    (pl.col("rank") * n_quintiles // n_available + 1).clip(1, n_quintiles).alias("quintile")
)

print(f"Quintile sizes:")
for q in range(1, n_quintiles + 1):
    q_df = pairs_with_q.filter(pl.col("quintile") == q)
    print(f"  Q{q}: {len(q_df)} pairs, p-value range [{q_df['p_value'].min():.4f}, {q_df['p_value'].max():.4f}]")

In [ ]:
h9_results = []

for q in tqdm(range(1, n_quintiles + 1), desc="H9: quintile backtests"):
    q_pairs = pairs_with_q.filter(pl.col("quintile") == q).drop(["rank", "quintile"])
    n_q = len(q_pairs)
    if n_q == 0:
        continue

    bt = PortfolioBacktester(max_pairs=n_q)
    result = bt.backtest(
        q_pairs,
        timeframe=DEFAULT_TF,
        start_date=TRADING_START,
        end_date=TRADING_END,
        max_pairs_to_trade=n_q,
    )
    m = result.metrics
    h9_results.append({
        "quintile": f"Q{q}",
        "n_pairs": n_q,
        "mean_p_value": float(q_pairs["p_value"].mean()),
        "sharpe": m["sharpe_ratio"],
        "win_rate": m["win_rate"],
        "avg_pnl": m["avg_pnl_net"],
        "total_return_pct": m["total_return_pct"],
        "num_trades": m["num_trades"],
    })

h9_df = pl.DataFrame(h9_results)
print(h9_df)

In [ ]:
h9_pd = h9_df.to_pandas()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Sharpe by Quintile", "Win Rate by Quintile"])
fig.add_trace(go.Bar(x=h9_pd["quintile"], y=h9_pd["sharpe"], name="Sharpe"), row=1, col=1)
fig.add_trace(go.Bar(x=h9_pd["quintile"], y=h9_pd["win_rate"], name="Win Rate"), row=1, col=2)
fig.update_layout(title="H9: P-Value as Quality Signal", height=400)
fig.update_yaxes(tickformat=".0%", row=1, col=2)
fig.show()

# Test monotonic relationship: Spearman rank correlation
if len(h9_pd) >= 3:
    rho, rho_p = stats.spearmanr(
        h9_pd["mean_p_value"], h9_pd["sharpe"]
    )
    print(f"\nSpearman correlation (p-value vs Sharpe): rho={rho:.3f}, p={rho_p:.4f}")
    if rho < 0 and rho_p < 0.05:
        print("RESULT: Significant negative correlation — lower p-value predicts higher Sharpe.")
    elif rho < 0:
        print("RESULT: Negative trend but not statistically significant.")
    else:
        print("RESULT: No evidence that p-value predicts profitability.")

---
## Summary

Aggregate results across all hypotheses.

In [ ]:
print("=" * 70)
print("HYPOTHESIS TESTING RESULTS SUMMARY")
print("=" * 70)

print(f"\nH1: Within-Sector Cointegration")
print(f"    Within-sector pass rate: {within_pass_rate:.1%} vs cross-sector: {cross_pass_rate:.1%}")
print(f"    Chi-square p = {chi2_p:.4e}, Odds ratio = {odds_ratio:.2f}")

best_h2 = h2_df.sort("sharpe", descending=True).row(0, named=True)
print(f"\nH2: Z-Score Entry Threshold")
print(f"    Best z_entry = {best_h2['z_entry']} (Sharpe = {best_h2['sharpe']:.3f})")

print(f"\nH3: Transaction Cost Sensitivity")
print(f"    Sensitivity: {slope:.4f} Sharpe/bps")

print(f"\nH4: IS vs OOS Degradation")
if sharpe_is != 0:
    print(f"    IS Sharpe = {sharpe_is:.3f}, OOS Sharpe = {sharpe_oos:.3f}")
    print(f"    Degradation = {degradation:.1f}%")

n_stable = len(h5_df.filter(pl.col("stable")))
n_tracked = len(h5_df)
print(f"\nH5: Half-Life Stability")
print(f"    {n_stable}/{n_tracked} pairs stable (CV < 0.5)")

best_h6 = h6_df.sort("sharpe", descending=True).row(0, named=True)
print(f"\nH6: Stop Loss Effectiveness")
print(f"    Best z_stop = {best_h6['z_stop_label']} (Sharpe = {best_h6['sharpe']:.3f})")

best_h7 = h7_df.sort("sharpe", descending=True).row(0, named=True)
print(f"\nH7: Portfolio Diversification")
print(f"    Best n_pairs = {best_h7['n_pairs']} (Sharpe = {best_h7['sharpe']:.3f})")

print(f"\nH8: Walk-Forward Robustness")
print(f"    Mean OOS Sharpe = {sharpes.mean():.3f} ± {sharpes.std():.3f}")
print(f"    % positive periods = {(sharpes > 0).mean():.0%}")

if len(h9_pd) >= 2:
    print(f"\nH9: P-Value as Quality Signal")
    print(f"    Q1 Sharpe = {h9_pd.iloc[0]['sharpe']:.3f}, Q{n_quintiles} Sharpe = {h9_pd.iloc[-1]['sharpe']:.3f}")

print("\n" + "=" * 70)